# Diffraction CPU C++ vs CUDA Benchmark

Цель: запускать отдельные `CPU C++` и `CUDA` backend'ы из ноутбука на одинаковых параметрах и сравнивать:
- время сборки матрицы
- время решения СЛАУ
- полное время backend'а
- совпадение коэффициентов разложения

`CPU C++` здесь намеренно вынесен отдельно от `CUDA`, чтобы использовать его как baseline для последующего сравнения ускорения GPU.


## План эксперимента

1. Собрать `DiffractionCpu.exe` и, при наличии CUDA toolchain, `DiffractionCuda.exe`.
2. Выполнить sanity-check на одном контрольном наборе параметров.
3. Прогнать sweep по `N`, углу падения и толщине скин-слоя.
4. Сравнить `CPU C++` и `CUDA` по времени и по максимальному расхождению коэффициентов.


In [ ]:
from __future__ import annotations

import itertools
import math
import statistics
import subprocess
from pathlib import Path

try:
    import pandas as pd
except ImportError:
    pd = None

REPO_ROOT = Path.cwd().resolve()
CPU_BUILD_SCRIPT = REPO_ROOT / 'Diffraction.Cpp' / 'build_cpu.bat'
CPU_EXE = REPO_ROOT / 'Diffraction.Cpp' / 'build' / 'DiffractionCpu.exe'
CUDA_BUILD_SCRIPT = REPO_ROOT / 'Diffraction.Cuda' / 'build_cuda.bat'
CUDA_EXE = REPO_ROOT / 'Diffraction.Cuda' / 'build' / 'DiffractionCuda.exe'

print('REPO_ROOT =', REPO_ROOT)
print('CPU build script exists =', CPU_BUILD_SCRIPT.exists())
print('CUDA build script exists =', CUDA_BUILD_SCRIPT.exists())


In [ ]:
DEFAULT_CASE = {
    'alpha1': -1.5,
    'beta1': -0.5,
    'alpha2': 0.5,
    'beta2': 1.5,
    'lambda': 1.0,
    'theta_deg': 10.0,
    'n': 10,
    'skin_depth': 0.001,
}

N_VALUES = [10, 20, 40]
ANGLE_VALUES_DEG = [0.0, 30.0, 60.0]
SKIN_DEPTH_VALUES = [0.0, 0.001, 0.01]
REPEATS = 3


In [ ]:
def run_command(command, cwd: Path = REPO_ROOT, check: bool = True):
    completed = subprocess.run(
        command,
        cwd=str(cwd),
        capture_output=True,
        text=True,
        shell=False,
    )
    if check and completed.returncode != 0:
        raise RuntimeError(
            f'Command failed: {command}\nSTDOUT:\n{completed.stdout}\nSTDERR:\n{completed.stderr}'
        )
    return completed


def build_backend(script_path: Path, required: bool):
    if not script_path.exists():
        if required:
            raise FileNotFoundError(script_path)
        return {'ok': False, 'stdout': '', 'stderr': f'Missing script: {script_path}'}

    result = run_command(['cmd', '/c', str(script_path)], check=False)
    ok = result.returncode == 0
    if required and not ok:
        raise RuntimeError(result.stderr or result.stdout)
    return {'ok': ok, 'stdout': result.stdout, 'stderr': result.stderr}


def parse_solver_output(text: str):
    parsed = {
        'status': None,
        'backend': None,
        'assembly_ms': None,
        'solve_ms': None,
        'total_ms': None,
        'coefficients': [],
        'message': None,
    }
    coeff_map = {}
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip()
        if key.startswith('coeff_'):
            idx = int(key.split('_', 1)[1])
            re_text, im_text = value.split(',')
            coeff_map[idx] = complex(float(re_text), float(im_text))
        elif key in {'assembly_ms', 'solve_ms', 'total_ms'}:
            parsed[key] = float(value)
        else:
            parsed[key] = value

    parsed['coefficients'] = [coeff_map[i] for i in sorted(coeff_map)]
    return parsed


def build_solver_args(case: dict):
    theta_rad = math.radians(case['theta_deg'])
    return [
        '--alpha1', str(case['alpha1']),
        '--beta1', str(case['beta1']),
        '--alpha2', str(case['alpha2']),
        '--beta2', str(case['beta2']),
        '--lambda', str(case['lambda']),
        '--theta', repr(theta_rad),
        '--n', str(case['n']),
        '--skin-depth', str(case['skin_depth']),
    ]


def run_solver(executable: Path, case: dict):
    result = run_command([str(executable), *build_solver_args(case)], check=False)
    parsed = parse_solver_output(result.stdout)
    parsed['returncode'] = result.returncode
    parsed['stdout'] = result.stdout
    parsed['stderr'] = result.stderr
    return parsed


def max_coeff_diff(cpu_coeffs, gpu_coeffs):
    if not cpu_coeffs or not gpu_coeffs or len(cpu_coeffs) != len(gpu_coeffs):
        return None
    return max(abs(a - b) for a, b in zip(cpu_coeffs, gpu_coeffs))


In [ ]:
cpu_build = build_backend(CPU_BUILD_SCRIPT, required=True)
cuda_build = build_backend(CUDA_BUILD_SCRIPT, required=False)

print('CPU build ok   =', cpu_build['ok'])
print('CUDA build ok  =', cuda_build['ok'])
if not cuda_build['ok']:
    print('CUDA build stderr/stdout:')
    print(cuda_build['stderr'] or cuda_build['stdout'])


In [ ]:
assert CPU_EXE.exists(), f'CPU executable not found: {CPU_EXE}'

cpu_sample = run_solver(CPU_EXE, DEFAULT_CASE)
print('CPU sample status =', cpu_sample['status'])
print('CPU sample backend =', cpu_sample['backend'])
print('CPU sample total_ms =', cpu_sample['total_ms'])

if cuda_build['ok'] and CUDA_EXE.exists():
    cuda_sample = run_solver(CUDA_EXE, DEFAULT_CASE)
    print('CUDA sample status =', cuda_sample['status'])
    print('CUDA sample backend =', cuda_sample['backend'])
    print('CUDA sample total_ms =', cuda_sample['total_ms'])
    print('Max coefficient abs diff =', max_coeff_diff(cpu_sample['coefficients'], cuda_sample['coefficients']))
else:
    cuda_sample = None
    print('CUDA sample skipped.')


In [ ]:
def benchmark_case(case: dict, repeats: int = REPEATS):
    rows = []
    cpu_runs = [run_solver(CPU_EXE, case) for _ in range(repeats)]
    rows.append({
        **case,
        'backend': 'CPU C++',
        'assembly_ms_mean': statistics.mean(run['assembly_ms'] for run in cpu_runs),
        'solve_ms_mean': statistics.mean(run['solve_ms'] for run in cpu_runs),
        'total_ms_mean': statistics.mean(run['total_ms'] for run in cpu_runs),
        'status': cpu_runs[-1]['status'],
        'max_coeff_abs_diff_vs_cpu': 0.0,
    })

    if cuda_build['ok'] and CUDA_EXE.exists():
        cuda_runs = [run_solver(CUDA_EXE, case) for _ in range(repeats)]
        rows.append({
            **case,
            'backend': 'CUDA',
            'assembly_ms_mean': statistics.mean(run['assembly_ms'] for run in cuda_runs),
            'solve_ms_mean': statistics.mean(run['solve_ms'] for run in cuda_runs),
            'total_ms_mean': statistics.mean(run['total_ms'] for run in cuda_runs),
            'status': cuda_runs[-1]['status'],
            'max_coeff_abs_diff_vs_cpu': max_coeff_diff(cpu_runs[-1]['coefficients'], cuda_runs[-1]['coefficients']),
        })

    return rows


cases = []
for n, theta_deg, skin_depth in itertools.product(N_VALUES, ANGLE_VALUES_DEG, SKIN_DEPTH_VALUES):
    case = dict(DEFAULT_CASE)
    case['n'] = n
    case['theta_deg'] = theta_deg
    case['skin_depth'] = skin_depth
    cases.append(case)

print(f'Total benchmark cases: {len(cases)}')


In [ ]:
benchmark_rows = []
for index, case in enumerate(cases, start=1):
    print(f'[{index}/{len(cases)}] n={case["n"]}, theta={case["theta_deg"]}, skin={case["skin_depth"]}')
    benchmark_rows.extend(benchmark_case(case))

if pd is not None:
    benchmark_df = pd.DataFrame(benchmark_rows)
    benchmark_df
else:
    benchmark_df = benchmark_rows
    benchmark_rows[:5]


In [ ]:
if pd is not None:
    summary = (
        benchmark_df
        .groupby(['backend', 'n'])[['assembly_ms_mean', 'solve_ms_mean', 'total_ms_mean']]
        .mean()
        .reset_index()
    )
    display(summary)

    if {'CPU C++', 'CUDA'}.issubset(set(benchmark_df['backend'])):
        cpu_df = benchmark_df[benchmark_df['backend'] == 'CPU C++'].copy()
        cuda_df = benchmark_df[benchmark_df['backend'] == 'CUDA'].copy()
        merged = cpu_df.merge(
            cuda_df,
            on=['alpha1', 'beta1', 'alpha2', 'beta2', 'lambda', 'theta_deg', 'n', 'skin_depth'],
            suffixes=('_cpu', '_cuda')
        )
        merged['speedup_total_cpu_over_cuda'] = merged['total_ms_mean_cpu'] / merged['total_ms_mean_cuda']
        merged['speedup_solve_cpu_over_cuda'] = merged['solve_ms_mean_cpu'] / merged['solve_ms_mean_cuda']
        display(merged[[
            'theta_deg', 'n', 'skin_depth',
            'total_ms_mean_cpu', 'total_ms_mean_cuda', 'speedup_total_cpu_over_cuda',
            'solve_ms_mean_cpu', 'solve_ms_mean_cuda', 'speedup_solve_cpu_over_cuda',
            'max_coeff_abs_diff_vs_cpu_cuda'
        ]])
else:
    print('Install pandas to get grouped tables.')


## Что смотреть дальше

- Увеличить `N_VALUES` и `REPEATS`, когда понадобится более стабильная статистика.
- При необходимости отдельно добавить sweep по длине волны и геометрии пластин.
- Если цель именно speedup GPU, сравнивать прежде всего `solve_ms_mean` и `assembly_ms_mean`, а не только `total_ms_mean`.
